# TICKET MJD-601
De: Paula Herrero · Marketing y tienda online

En el MJD-505 encontrasteis los pedidos web que no llegaron al ERP. Ahora quiero el cuadre completo:
pedido a pedido, ¿la tienda web y el ERP dicen lo mismo? Mismos pedidos, mismo estado, mismo impor‐
te.

# CRITERIOS
- Todos los pedidos de pedidos_web.json cruzados con los del ERP, en las dos direcciones.
- Los estados de la web traducidos a los del ERP, y cuántos no coinciden.
- Los pedidos con importe distinto, cuántos son y por qué.

In [15]:
import os, sys
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [7]:
import json
import duckdb
import pandas as pd
from pathlib import Path

RAICES = (Path.cwd(), *Path.cwd().parents)
RAIZ = next((p for p in RAICES if (p / "datos/web/pedidos_web.json").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError(f"No se encontró datos/web/pedidos_web.json desde {Path.cwd()}")

with open(RAIZ / "datos/web/pedidos_web.json", encoding="utf-8") as f:
    web = json.load(f)
cab = pd.json_normalize(web["orders"]).drop(columns="items")
lin = pd.json_normalize(web["orders"], record_path="items", meta=["order_id"])
lin["importe"] = lin["qty"] * lin["unit_price"] * (1 - lin["discount"] / 100)
cab = cab.merge(
    lin.groupby("order_id", as_index=False)["importe"].sum(),
    on="order_id", how="left", validate="one_to_one",
)

cab["erp_pedido_id"] = cab["erp_pedido_id"].astype("Int64")

con = duckdb.connect(str(RAIZ / "db/marejada.duckdb"), read_only=True)
erp = con.sql("""
    SELECT p.pedido_id, p.estado, coalesce(sum(l.importe_neto), 0) AS importe_erp
    FROM silver.pedidos AS p
    LEFT JOIN silver.lineas_pedido AS l USING (pedido_id)
    WHERE p.canal = 'Web B2B'
    GROUP BY p.pedido_id, p.estado
""").df()
con.close()

print(cab.shape, erp.shape)
print(cab["erp_pedido_id"].isna().sum())

(7956, 8) (7925, 3)
31


PEDIDOS EN LAS DOS DIRECCIONES:

In [8]:
assert cab["erp_pedido_id"].dropna().is_unique, "un pedido del ERP aparece dos veces en la web"
cruce = cab.merge(
erp, left_on="erp_pedido_id", right_on="pedido_id",
how="outer", indicator=True, validate="many_to_one",
)
print(cruce["_merge"].value_counts())
cruce.loc[cruce["_merge"] == "left_only", "status"].value_counts()

_merge
both          7925
left_only       31
right_only       0
Name: count, dtype: int64


status
sync_error    31
Name: count, dtype: int64

ESTADOS

In [9]:
traduccion = {"completed": ["Entregado", "Devuelto parcial"], "cancelled": ["Cancelado"],
"processing": ["Pendiente"], "shipped": ["En reparto"]}
ambos = cruce[cruce["_merge"] == "both"].copy()
ambos["estado_ok"] = [e in traduccion.get(s, []) for s, e in zip(ambos["status"], ambos["estado"])]
print(ambos["estado_ok"].value_counts())
pd.crosstab(ambos["status"], ambos["estado"])

estado_ok
True    7925
Name: count, dtype: int64


estado,Cancelado,Devuelto parcial,En reparto,Entregado,Pendiente
status,,,,,
cancelled,297,0,0,0,0
completed,0,107,0,7436,0
processing,0,0,0,0,36
shipped,0,0,49,0,0


IMPORTES

In [10]:
ambos["dif"] = (ambos["importe"] - ambos["importe_erp"]).round(2)
distintos = ambos[ambos["dif"].abs() > 0.01]
print(len(distintos))
print(distintos["estado"].value_counts())
print(distintos["dif"].describe().round(2))

107
estado
Devuelto parcial    107
Name: count, dtype: int64
count    107.00
mean      63.64
std       78.89
min        4.79
25%       19.18
50%       37.07
75%       75.65
max      475.61
Name: dif, dtype: float64


Los pedidos con importe distinto son exactamente los «Devuelto parcial», y en todos la web dice más que el
ERP. La explicación está en las líneas: el ERP registra la devolución como líneas negativas dentro del mismo
pedido (Volumen 02), y la web no. Segunda vía de comprobación: si se quitan las líneas negativas del ERP,
los importes deberían cuadrar.

In [14]:
con = duckdb.connect(str(RAIZ / "db/marejada.duckdb"), read_only=True)
sin_devol = con.sql("""
SELECT pedido_id, sum(importe_neto) AS importe_sin_devoluciones
FROM silver.lineas_pedido
WHERE cantidad > 0
GROUP BY pedido_id
""").df()
con.close()
check = distintos.merge(sin_devol, on="pedido_id", validate="one_to_one")
((check["importe"] - check["importe_sin_devoluciones"]).abs() > 0.01).sum()

np.int64(0)